In [1]:
from src.classes.crossword_puzzle import CrosswordPuzzle
from src.classes.guesses import Guess
from src.constants import CLUE_ID, PUZ_FILE_DIR
from src.prompts.get_guesses_for_clue_using_llm import get_guesses_for_clue_using_llm
from src.prompts.reorder_clues_using_llm import reorder_clues_using_llm

In [2]:
file = f"{PUZ_FILE_DIR}/Goobix_4x4Puzzle1.puz"
crossword_puzzle = CrosswordPuzzle(file)
clues = reorder_clues_using_llm(crossword_puzzle.get_clues(), debug=True)
guesses: dict[CLUE_ID, list[Guess]] = {}

=== REORDER CLUES with gemma4:e4b (Attempts: 1) ===

You are a crossword puzzle solver. You are given a list of crossword clues.
Assign each clue a difficulty score from 1 to 100, with 1 being the easiest and 100 being the hardest.
Provide a brief explanation of why each clue is considered to have the given difficulty score,
including any wordplay, obscurity, or other factors that contribute to its difficulty.
If you cannot score a clue confidently, omit it so it can be retried separately.

Clues:
- (1 across): Some drink it when they want to have fun.
- (5 across): Special kind of music.
- (6 across): Continent in the East of Europe.
- (7 across): To scream.
- (1 down): The loud cry of a mule or donkey.
- (2 down): Rest.
- (3 down): Bad.
- (4 down): Which is not fake.

=== REORDERED CLUES gemma4:e4b ===
  1 down: The loud cry of a mule or donkey. (Difficulty: 10)
  3 down: Bad. (Difficulty: 10)
  7 across: To scream. (Difficulty: 15)
  2 down: Rest. (Difficulty: 15)
  6 across: Contin

In [3]:
clues

[Clue(text='The loud cry of a mule or donkey.', length=4, number=1, direction='down', row=0, col=0),
 Clue(text='Bad.', length=4, number=3, direction='down', row=0, col=2),
 Clue(text='To scream.', length=4, number=7, direction='across', row=3, col=0),
 Clue(text='Rest.', length=4, number=2, direction='down', row=0, col=1),
 Clue(text='Continent in the East of Europe.', length=4, number=6, direction='across', row=2, col=0),
 Clue(text='Some drink it when they want to have fun.', length=4, number=1, direction='across', row=0, col=0),
 Clue(text='Special kind of music.', length=4, number=5, direction='across', row=1, col=0),
 Clue(text='Which is not fake.', length=4, number=4, direction='down', row=0, col=3)]

In [4]:
clue_index = 0

while not crossword_puzzle.is_solved:
    clue = clues[clue_index]
    print(f"Attempting to solve clue {clue.number} {clue.direction} - {clue.text}")

    if clue.id not in guesses:
        pattern = crossword_puzzle.get_pattern(clue)
        print(f"Generating guesses for clue: {clue.number} {clue.direction} - {clue.text} with pattern '{pattern}'")
        clue_guesses = get_guesses_for_clue_using_llm(clue, pattern, debug=True)
        guesses[clue.id] = clue_guesses

    clue_guesses = guesses[clue.id]

    if len(clue_guesses) == 0:
        print(f"No guesses left for clue: {clue.number} {clue.direction} - {clue.text}, backtracking...")
        guesses.pop(clue.id)

        if clue_index != 0:
            clue_index -= 1
            crossword_puzzle.remove_answer(clues[clue_index])

        continue

    print(f"Guesses for clue {clue.number} {clue.direction}:")
    for guess in clue_guesses:
        print(f" - {guess.answer} (confidence: {guess.confidence_score}): {guess.explanation}")

    best_guess = max(clue_guesses, key=lambda g: g.confidence_score)

    try:
        crossword_puzzle.set_answer(clue, best_guess.answer)
        clue_guesses.remove(best_guess)

        print(f"Set answer for clue {clue.number} {clue.direction} to '{best_guess.answer}'")
        clue_index += 1
    except Exception as e:

        clue_guesses.remove(best_guess)
        print(f"Error setting answer for clue {clue.number} {clue.direction}: {e}")
    finally:
        crossword_puzzle.print_grid()

Attempting to solve clue 1 down - The loud cry of a mule or donkey.
Generating guesses for clue: 1 down - The loud cry of a mule or donkey. with pattern '['_', '_', '_', '_']'
=== GENERATE GUESSES with gemma4:e4b ===
You are an expert crossword solver. Provide five unique guesses that satisfy the structural and semantic requirements below. If no valid guesses exist that meet all criteria, return an empty list: [].

### CROSSWORD RULES
The following are rules for generating valid guesses from most to least important. Prioritize guesses that satisfy more of these rules, and assign higher confidence scores to those guesses accordingly.
1. RELEVANCE: Guesses must be directly be related to the clue and fit in the majority on contexts. DO NOT include guesses that are only remotely related to the clue or require multiple leaps of logic to connect them to the clue's meaning.
2. PATTERN MATCHING: Guesses must fit the specified pattern, matching known letters in their exact positions and having 